This notebook is dedicated to cleaning and merging the three data sources 
needed to construct the analysis panel.

The Median Multiple — our primary predictor — will be constructed by dividing 
Zillow median home values by ACS median household income at the county-year level.

Outcome and mediator variables will be extracted from County Health Rankings 
data and column naming conventions standardized across years.

## CHR Cleaning Tasks
- Load correct sheet ('Select Measure Data' for 2024, 'Ranked Measure Data' for all others)
- Use header=[0,1] for MultiIndex
- Drop state row (index 0, FIPS 6000)
- Standardize first-level column names to title case (2016–2022 → 2023–2024 convention)
- Select target columns (FIPS, County, plus 5 variables)
- Rename second-level column names to consistent standard names
- Add year column
- Stack all years into one long dataframe


In [ ]:
# imports

import pandas as pd
import numpy as np
import os

os.chdir('/Users/douglas/Documents/GitHub/SDOH_Housing_Health_CA/notebooks')
print(os.getcwd())

Dramatic variance within Preventable Hospital Stay values were observed between years (2016 ~35-46 vs 2024 ~1774-3939). Methodology was investigated and a unit change was discovered between 2018-2019.
The hospital stays were calculated as follows:
* 2016 - 2018 was based on a per 1000 metric
* 2019 - 2024 is based on a per 100,000 metric

The values within 2016-2018 will be multiplied by 100 in order to standardize the metric for analysis.

In [ ]:
# Load county health rankings data
files = os.listdir('../data/raw/county_health_rankings')
chr_frames = []

# Loop through all CHR files 2016-2024, skipping 2025
for f in sorted(files):
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    # 2024 uses a different sheet name than all prior years
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    year = f[:4]
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    # Drop state-level row (FIPS 6000) and empty rows
    df_year = df_year.drop(index=0)
    df_year = df_year[df_year[('Unnamed: 0_level_0', 'FIPS')].notna()]
    # Standardize first-level column names to title case (2016-2022 use lowercase)
    df_year.columns = [(_col[0].title(), _col[1]) for _col in df_year.columns]
    # Second-level column names changed between years — set correct names per year
    if '2016' in f or '2017' in f or '2018' in f or '2019' in f:
        mental_health_col = 'Mentally Unhealthy Days'
        preventable_col = 'Preventable Hosp. Rate'
        lbw_col = '% LBW'
    elif '2021' in f or '2022' in f:
        lbw_col = '% Low birthweight'
        mental_health_col = 'Average Number of Mentally Unhealthy Days'
        preventable_col = 'Preventable Hospitalization Rate'
    else:
        mental_health_col = 'Average Number of Mentally Unhealthy Days'
        preventable_col = 'Preventable Hospitalization Rate'
        lbw_col = '% Low Birthweight'
    # Select target columns and rename to consistent standard names
    df_col_select = df_year[[('Unnamed: 0_Level_0', 'FIPS'),('Unnamed: 2_Level_0', 'County'),
        ('Premature Death', 'Years of Potential Life Lost Rate'),
        ('Poor Mental Health Days', mental_health_col),
        ('Preventable Hospital Stays', preventable_col),
        ('Children In Poverty', '% Children in Poverty'),
        ('Low Birthweight', lbw_col)]].copy()
    df_col_select.columns = ['fips', 'county', 'premature_death_rate',
        'poor_mental_health_days', 'preventable_hosp_rate',
        'children_in_poverty_pct', 'low_birthweight_pct']
    df_col_select['year'] = f[:4]
    # Standardize preventable hospital stays: 2016-2018 reported per 1,000, 
    # 2019+ reported per 100,000 — multiply older values by 100
    if df_col_select['year'].iloc[0] in ['2016', '2017', '2018']:
        df_col_select['preventable_hosp_rate'] = df_col_select['preventable_hosp_rate'] * 100
    chr_frames.append(df_col_select)

chr_panel = pd.concat(chr_frames, ignore_index=True)
print('CHR panel shape:', chr_panel.shape)

## CHR Panel — Complete

The CHR cleaning pipeline produced a panel of 522 rows (58 counties × 9 years, 
2016–2024). Column naming inconsistencies were resolved across all years. 
Preventable Hospital Stays values were standardized to a per 100,000 metric. 
The state-level row and empty rows were dropped. Two rural counties (Alpine, 
Sierra) have suppressed Premature Death values due to small population — 
handled via listwise deletion at the model level.

In [3]:
chr_panel.to_csv('../data/processed/chr_panel.csv', index=False)

## ACS Cleaning Pipeline

The ACS files contain median household income data for all US counties. 
Each year is stored as a separate CSV file. The following cleaning steps 
are required for each year:

- Load the Data file (header=0, single row header)
- Filter to California counties using GEO_ID starting with '0500000US06'
- Select only GEO_ID and 'S1901_C01_012E' (Median Household Income Estimate)
- Extract year from filename and add as a column
- Handle suppression codes '(X)' and 'N' — replace with NaN
- Stack all years into one long dataframe
- Note: 2020 is missing due to COVID-19 data collection disruption

## Data Source Limitation: Rural County Income Data

During the data cleaning phase a significant data limitation was discovered. 
The ACS 1-year estimates exclude counties with populations below 65,000, 
resulting in 18 of 58 California counties (~31%) having no annual household 
income data available.

Missing counties: Alpine, Amador, Calaveras, Colusa, Del Norte, Glenn, Inyo, 
Lassen, Mariposa, Modoc, Mono, Plumas, San Benito, Sierra, Siskiyou, Tehama, 
Trinity, and Tuolumne.

For the purpose of this analysis, ACS 5-year estimates will be substituted 
for these counties. The 5-year estimate for a given year represents a smoothed 
average across a 5-year window centered on that year, and is therefore less 
precise than an annual estimate. Rows using 5-year estimates will be flagged 
with an `income_source` column ('1yr' or '5yr') to maintain transparency.

Source: U.S. Census Bureau, American Community Survey 5-Year Estimates, 
Table S1901. https://data.census.gov

**Planned improvement:** A future iteration will implement imputation by 
anchoring on 5-year CA estimates and informing annual growth rates through 
random sampling of rural counties (population < 65,000) from comparable 
coastal states. Imputed values will be drawn from a normal distribution 
parameterized by the mean and variance of the proxy county growth rates.

In [5]:
# The 18 rural California counties missing from ACS 1-year estimates
# (population < 65,000, suppressed by Census Bureau)
required_counties = [
    'Alpine County, California', 'Amador County, California',
    'Calaveras County, California', 'Colusa County, California',
    'Del Norte County, California', 'Glenn County, California',
    'Inyo County, California', 'Lassen County, California',
    'Mariposa County, California', 'Modoc County, California',
    'Mono County, California', 'Plumas County, California',
    'San Benito County, California', 'Sierra County, California',
    'Siskiyou County, California', 'Tehama County, California',
    'Trinity County, California', 'Tuolumne County, California'
]

In [ ]:
# Verify all 18 rural counties are present in 5-year ACS data
file_path = '../data/raw/census_acs_5yr/ACSST5Y2020.S1901-Data.csv'
df_2020 = pd.read_csv(file_path, header=0)
df_2020 = df_2020.drop(index=0)

# Filter to CA counties
ca_5yr = df_2020[
    (df_2020['GEO_ID'].str.startswith('0500000US06')) &
    (df_2020['GEO_ID'].str.len() == 14)
]

# Confirm all 18 missing counties are present in 5-year file
found = ca_5yr['NAME'].tolist()
for county in required_counties:
    if county in found:
        print(f'✓ {county}')
    else:
        print(f'MISSING: {county}')

In [ ]:
# Loops for combining 1 and 5 year census_acs data

acs_frames = []

# 1-year loop
files_1yr = os.listdir('../data/raw/census_acs')
for f in sorted(files_1yr):
    if f.endswith('-Data.csv'):
        income_source = '1yr'
        file_path = '../data/raw/census_acs/' + f
        year = f[7:11]      
        df_year = pd.read_csv(file_path, header=[0])
        df_year = df_year.drop(index=0)
        df_year = df_year.drop_duplicates()
        df_year = df_year.replace({'(X)': None, 'N': None})
        df_year['year'] = year
        # Filter to California counties
        df_year = df_year[
                (df_year['GEO_ID'].str.startswith('0500000US06')) & 
                (df_year['GEO_ID'].str.len() == 14)] 
        # create a selection df with data of interest
        df_col_select = df_year[['year','NAME', 'GEO_ID','S1901_C01_012E']].copy()
        df_col_select['income_source'] = income_source
        acs_frames.append(df_col_select)
        print(df_col_select.shape)

# 5-year loop  
files_5yr = os.listdir('../data/raw/census_acs_5yr')
for f in sorted(files_5yr):
    if f.endswith('-Data.csv'):
        income_source = '5yr'
        file_path = '../data/raw/census_acs_5yr/' + f
        year = f[7:11]      
        df_year = pd.read_csv(file_path, header=[0])
        df_year = df_year.drop(index=0)
        df_year = df_year.drop_duplicates()
        df_year = df_year.replace({'(X)': None, 'N': None})
        df_year['year'] = year
        # Filter to required counties
        df_year = df_year[df_year['NAME'].isin(required_counties)]
        # create a selection df with data of interest
        df_col_select = df_year[['year','NAME', 'GEO_ID','S1901_C01_012E']].copy()
        df_col_select['income_source'] = income_source
        acs_frames.append(df_col_select)
        print(df_col_select.shape)
        

acs_panel = pd.concat(acs_frames, ignore_index=True)

In [25]:
# Sort so '1yr' comes before '5yr' alphabetically — ensures 1-year estimate 
# is kept when a county appears in both sources (e.g. Tehama in later years)

acs_panel = acs_panel.sort_values('income_source')
acs_panel = acs_panel.drop_duplicates(subset=['NAME', 'year'], keep='first')

In [ ]:
print(acs_panel.shape)
print(acs_panel['year'].value_counts().sort_index())

In [ ]:
print(acs_panel['income_source'].value_counts())

In [14]:
# save acs_panel
acs_panel.to_csv('../data/processed/acs_panel.csv', index=False)

## ACS Panel — Complete

The ACS pipeline combined 1-year estimates (329 rows, 40 large counties) 
and 5-year estimates (153 rows, 18 rural counties with population < 65,000). 
After deduplication, the final ACS panel contains 482 rows x 6 columns 
across 8 years (2020 absent for urban counties due to COVID-19 ACS suspension). 
Income source flagged per row ('1yr' or '5yr') for transparency.

## Zillow Home Value Data

The Zillow Home Value Index (ZHVI) provides monthly median home values at the 
county level for all US counties. The following cleaning steps are applied:

- Filter to California counties using the 'State' column
- Construct 5-digit FIPS code from StateCodeFIPS and MunicipalCodeFIPS
- Select monthly columns for analysis years 2016–2024 only
- Collapse 12 monthly values to a single annual average per county per year
- Reshape from wide format (one row per county, monthly columns) to long format 
  (one row per county per year)

Output: 522 rows × 4 columns (fips, RegionName, year, median_home_value)

In [ ]:
print(os.getcwd())

# load zillow file
file_path = '../data/raw/zillow/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv'

zillow_df = pd.read_csv(
    file_path,
    header= 0)

# Inspect file
print('Shape:', zillow_df.shape)
print('\nColumn names:')
for col in zillow_df.columns:
    print(col)

In [ ]:
zillow_df['StateName'].head(5)

In [ ]:
# load zillow file
file_path = '../data/raw/zillow/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv'

zillow_df = pd.read_csv(
    file_path,
    header= 0)

zillow_df = zillow_df[zillow_df['State'] == 'CA'] 
zillow_df['FIPS'] = zillow_df['StateCodeFIPS'].astype(str).str.zfill(2) + \
                    zillow_df['MunicipalCodeFIPS'].astype(str).str.zfill(3)
years = ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
date_cols = [col for col in zillow_df.columns if any(col.startswith(y) for y in years)]
zillow_df = zillow_df[['FIPS', 'RegionName'] + date_cols]
for y in years:
    zillow_df[y] = zillow_df[[col for col in zillow_df.columns if col.startswith(y)]].mean(axis=1)   

#  Inspect file
print('Shape:', zillow_df.shape)
zillow_df.head()


 

In [ ]:
# select the relevent columns 

zillow_annual = zillow_df[['FIPS', 'RegionName'] + years]
print(zillow_annual.shape)

In [ ]:
zillow_long = pd.melt(
    zillow_annual,
    id_vars=['FIPS', 'RegionName'],
    value_vars=years,
    var_name='year',
    value_name='median_home_value'
)

zillow_long

In [20]:
# Save the Zillow Panel
zillow_long.to_csv('../data/processed/zillow_panel.csv', index=False)

## Merge all panels into the same panel

In [ ]:
print(f'Zillow Panel\n {zillow_long.head(3)}')
print(f'County Health Panel\n {chr_panel.head(3)}')
print(f'ACS Panel\n {acs_panel.head(3)}')

In [ ]:
# Transform chr panel fips data to standard format
chr_panel['fips'] = chr_panel['fips'].astype(int).astype(str).str.zfill(5)
chr_panel['fips'].head(5)

In [ ]:
# Create a fips column for the ACS Panel using GEO_ID
acs_panel['fips'] = acs_panel['GEO_ID'].str[9:]
acs_panel['fips'].head()


In [ ]:
# Standarize fips column name to lowercase 

zillow_long = zillow_long.rename(columns={'FIPS': 'fips'})
zillow_long.columns

In [ ]:
# join all panel data 

chr_zillow = pd.merge(chr_panel, zillow_long, on=['fips', 'year'], how='left')
full_panel = pd.merge(chr_zillow, acs_panel, on=['fips', 'year'], how = 'left')

print(f'Shape\n {full_panel.shape}')
full_panel.head()

In [28]:
# drop unneeded columns and rename median household income column

full_panel = full_panel.drop(columns=['RegionName', 'NAME', 'GEO_ID'])
full_panel = full_panel.rename(columns={'S1901_C01_012E': 'median_household_income'})


In [ ]:
print(full_panel.shape)
full_panel.columns

In [ ]:
# construct the median multiple and verify it
full_panel['median_multiple'] = full_panel['median_home_value'] / full_panel['median_household_income'].astype(float)

full_panel['median_multiple'].head()

In [ ]:
print(full_panel.shape)
print(full_panel.isnull().sum())

In [ ]:
missing_income = full_panel[full_panel['median_household_income'].isna()]
print(missing_income['year'].value_counts())

Note on Methodology

2020 urban county income data is absent due to ACS 1-year estimate suspension during COVID-19. Given the economic disruption of 2020-2021, imputation was not attempted as it risked obscuring genuine pandemic-era income dynamics.

In [33]:
# save the full panel
full_panel.to_csv('../data/processed/full_panel.csv', index=False)